In [ ]:
from vllm import LLM, SamplingParams
import json
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "6"
model_id = "mistralai/Mistral-7B-Instruct-v0.2"
# 加载模型
llm = LLM(model=model_id, dtype="float16", max_model_len=4096)

/common/home/sl2148/anaconda3/envs/prune_llm_yang_310/lib/python3.10/site-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


WARNING 09-18 11:04:02 config.py:1454] Casting torch.bfloat16 to torch.float16.
INFO 09-18 11:04:02 llm_engine.py:174] Initializing an LLM engine (v0.5.4) with config: model='mistralai/Mistral-7B-Instruct-v0.3', speculative_config=None, tokenizer='mistralai/Mistral-7B-Instruct-v0.3', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None), seed=0, served_model_name=mistralai/Mistral-7B-Instruct-v0.3, use_v2_block_manager=False, enable_prefix_caching=False)
INFO 09-18 11:04:02 selector.

/common/home/sl2148/anaconda3/envs/prune_llm_yang_310/lib/python3.10/site-packages/xformers/ops/fmha/flash.py:211: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_fwd")
/common/home/sl2148/anaconda3/envs/prune_llm_yang_310/lib/python3.10/site-packages/xformers/ops/fmha/flash.py:344: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_bwd")


INFO 09-18 11:04:05 model_runner.py:720] Starting to load model mistralai/Mistral-7B-Instruct-v0.3...
INFO 09-18 11:04:05 selector.py:151] Cannot use FlashAttention-2 backend for Volta and Turing GPUs.
INFO 09-18 11:04:05 selector.py:54] Using XFormers backend.
INFO 09-18 11:04:05 weight_utils.py:225] Using model weights format ['*.safetensors']


Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


INFO 09-18 11:04:13 model_runner.py:732] Loading model weights took 13.5083 GB
INFO 09-18 11:04:14 gpu_executor.py:102] # GPU blocks: 2671, # CPU blocks: 2048
INFO 09-18 11:04:17 model_runner.py:1024] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 09-18 11:04:17 model_runner.py:1028] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 09-18 11:04:32 model_runner.py:1225] Graph capturing finished in 16 secs.


In [4]:

# 读取 GSM8K 数据
gsm8k_file = "../data/GSM8K_eval_build/eval_cot0shot.jsonl"
with open(gsm8k_file, "r") as f:
    data = [json.loads(line) for line in f]

# 取前几个样本测试
prompts = [d["input"] for d in data[:5]]

# 采样参数，强制尽快停下
sampling_params = SamplingParams(
    temperature=0.0,       # 固定输出，减少废话
    top_p=1.0,
    max_tokens=1024,         # 限制最多生成 64 tokens
    stop=["\n\n\n"]            # 一旦模型换行就停
)
B_INST, E_INST = "[INST]", "[/INST]"
B_SYS, E_SYS = "<<SYS>>\n", "\n<</SYS>>\n\n"
system_prompt = "You are a helpful assistant."
formatted_prompts = [
    f"{B_INST} {B_SYS} {system_prompt} {E_SYS} {prompt} {E_INST}"
    for prompt in prompts
]

# 生成
outputs = llm.generate(prompts, sampling_params)


# 打印结果（只保留第一行）
for output in outputs:
    q = output.prompt
    a = output.outputs[0].text.strip().split("\n")
    print(f"Q: {q}\nA: {a}\n")


Processed prompts:   0%|          | 0/5 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|██████████| 5/5 [00:14<00:00,  2.89s/it, est. speed input: 35.88 toks/s, output: 86.72 toks/s]

Q: Please act as a math teacher and solve the math problem step by step.
# Question:
Marcus, Humphrey, and Darrel are bird watching. Marcus sees 7 birds, Humphrey sees 11 birds, and Darrel sees 9 birds. How many birds does each of them see on average?
# Reasoning:
Let's think step by step.
A: ["1. First, let's add the total number of birds each person saw:", "Total birds = Marcus' birds + Humphrey's birds + Darrel's birds", 'Total birds = 7 + 11 + 9 = 27 birds', '', "2. Now, let's find the average number of birds each person saw. To do this, we'll divide the total number of birds by the number of people:", 'Average birds per person = Total birds / Number of people', 'Average birds per person = 27 / 3', '', "3. Since we can't divide by a fraction, we'll convert the fraction to a decimal by dividing the numerator (top number) by the denominator (bottom number):", 'Average birds per person = 27 ÷ 3 = 9', '', 'So, on average, each person saw 9 birds.']

Q: Please act as a math teacher and 